In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from tensorflow.keras.models import load_model
import pickle

model = load_model("/content/drive/MyDrive/brain_tumor_models/best_model_inceptionv3.h5")
class_names = pickle.load(open("/content/drive/MyDrive/brain_tumor_models/class_names.pkl", "rb"))


In [3]:
!cp /content/drive/MyDrive/brain_tumor_models/best_model_inceptionv3.h5 .
!cp /content/drive/MyDrive/brain_tumor_models/class_names.pkl .

In [4]:
%%writefile app.py

import streamlit as st
import tensorflow as tf
import numpy as np
import pickle
from PIL import Image

# ===============================
# Page Configuration
# ===============================
st.set_page_config(
    page_title="Brain Tumor Detection",
    page_icon="🧠",
    layout="centered"
)

st.title("🧠 Brain Tumor Detection System")
st.write(
    "Upload a **Brain MRI image** to predict the tumor type using a deep learning model."
)

# ===============================
# Load Model and Class Names
# ===============================
@st.cache_resource
def load_model_and_classes():
    model = tf.keras.models.load_model("best_model_inceptionv3.h5")
    with open("class_names.pkl", "rb") as f:
        class_names = pickle.load(f)
    return model, class_names

model, class_names = load_model_and_classes()
st.success("Model loaded successfully ✅")

# ===============================
# Image Preprocessing
# ===============================
IMG_SIZE = 299  # InceptionV3 input size

def preprocess_image(image):
    image = image.resize((IMG_SIZE, IMG_SIZE))
    image = np.array(image)

    # Handle grayscale images
    if image.ndim == 2:
        image = np.stack((image,) * 3, axis=-1)

    image = image / 255.0
    image = np.expand_dims(image, axis=0)
    return image

# ===============================
# File Upload
# ===============================
uploaded_file = st.file_uploader(
    "📤 Upload Brain MRI Image",
    type=["jpg", "jpeg", "png"]
)

if uploaded_file is not None:
    image = Image.open(uploaded_file).convert("RGB")

    st.image(image, caption="Uploaded MRI Image", width=700)

    if st.button("🔍 Predict Tumor"):
        with st.spinner("Analyzing MRI image..."):
            img_array = preprocess_image(image)
            predictions = model.predict(img_array)

        predicted_index = np.argmax(predictions[0])
        predicted_class = class_names[predicted_index]
        confidence = predictions[0][predicted_index] * 100

        # ===============================
        # Results
        # ===============================
        st.success("✅ Prediction Completed")

        st.subheader("🧾 Prediction Result")
        st.write(f"**Tumor Type:** `{predicted_class}`")
        st.write(f"**Confidence:** `{confidence:.2f}%`")

        st.progress(int(confidence))

        # ===============================
        # Class Probabilities
        # ===============================
        st.subheader("📊 Class Probabilities")

        sorted_indices = np.argsort(predictions[0])[::-1]
        for i in sorted_indices:
            st.write(f"{class_names[i]} : {predictions[0][i]*100:.2f}%")


Writing app.py


In [5]:
!pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 107.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 118.5 MB/s eta 0:00:00


In [11]:
!pip install streamlit streamlit_option_menu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.3/829.3 kB 42.4 MB/s eta 0:00:00


In [12]:
!ls

app.py			   class_names.pkl	    drive     sample_data
best_model_inceptionv3.h5  cloudflared-linux-amd64  logs.txt


In [13]:
!pip install -q streamlit
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64
import subprocess
subprocess.Popen(["./cloudflared-linux-amd64", "tunnel", "--url", "http://localhost:8501"])
!nohup /content/cloudflared-linux-amd64 tunnel --url http://localhost:8501 &

--2026-02-02 08:34:37--  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
Resolving github.com (github.com)... 140.82.113.4
Connecting to github.com (github.com)|140.82.113.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/cloudflare/cloudflared/releases/download/2026.1.2/cloudflared-linux-amd64 [following]
--2026-02-02 08:34:37--  https://github.com/cloudflare/cloudflared/releases/download/2026.1.2/cloudflared-linux-amd64
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/106867604/31ade8d6-2fcd-4925-9218-5534d27a01dc?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-02-02T09%3A15%3A30Z&rscd=attachment%3B+filename%3Dcloudflared-linux-amd64&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-02-02T0

In [14]:
!streamlit run /content/app.py &>/content/logs.txt &

In [15]:
!grep -o 'https://.*\.trycloudflare.com' nohup.out | head -n 1 | xargs -I {} echo "Your tunnel url {}"

Your tunnel url https://installed-burke-ministers-times.trycloudflare.com
